# 7.21 — Swin Transformer

Swin Transformer keeps the token-and-attention idea from ViT, but turns it into a practical vision backbone: patches become a 2-D grid, attention is restricted to local windows, alternate blocks shift those windows so information crosses borders, and patch merging builds a pyramid of coarser but richer features for detection and segmentation.

## 📖 Concept walkthrough — build each idea from scratch

Before the terse worked examples, we build Swin one idea at a time. Run each cell in order and read the printed intermediate values — the point is to see exactly how a transformer can behave like a hierarchical vision backbone while still using ordinary softmax attention. This walkthrough is self-contained (it imports what it needs) and uses a `_w` suffix on its variables so it never clashes with the examples below.

In [ ]:
import numpy as np  # arrays, reshaping, and attention math.
import matplotlib.pyplot as plt  # all walkthrough visualizations.
np.random.seed(0)  # reproducibility for toy projections and displays.

### 1. Patch tokens keep the image grid

Swin starts the same way as a Vision Transformer: split an image into non-overlapping patches and flatten each patch into a token. The difference is that Swin keeps track of the 2-D token grid, because later windows, shifts, and merging all depend on spatial neighborhoods. In this tiny example, a 4×4 image with 2×2 patches becomes a 2×2 grid of 4-dimensional tokens.

In [ ]:
img_w = np.arange(16).reshape(4, 4)  # a tiny grayscale image with readable pixel ids.
patch_w = 2  # each token will summarize a 2x2 block.
print("image:\n", img_w)
print("image shape:", img_w.shape, "patch size:", patch_w)

▶ What you'll see: a 4×4 image whose numbers make it easy to verify which pixels enter each patch.

In [ ]:
patches_w = img_w.reshape(2, patch_w, 2, patch_w).transpose(0, 2, 1, 3)  # grid_y, grid_x, patch_y, patch_x.
tokens_w = patches_w.reshape(2, 2, patch_w * patch_w)  # keep a 2-D token grid, not just a flat sequence.
print("patch grid shape:", tokens_w.shape)
print("top-left token:", tokens_w[0, 0])
print("bottom-right token:", tokens_w[1, 1])
assert tokens_w.shape == (2, 2, 4)
assert np.all(tokens_w[0, 0] == np.array([0, 1, 4, 5]))

▶ What you'll see: four patch tokens; the first is `[0, 1, 4, 5]`, exactly the top-left 2×2 image block.

In [ ]:
plt.figure(figsize=(4, 3))
plt.imshow(img_w, cmap="viridis")
for cut_w in [1.5]:
    plt.axhline(cut_w, color="white", linewidth=2)
    plt.axvline(cut_w, color="white", linewidth=2)
plt.title("1: 4x4 image split into 2x2 patches")
plt.colorbar(label="pixel id")
plt.show()

▶ What you'll see: white grid lines mark the four patches that become Swin's first tokens.

*Why it's done this way:* attention works on tokens, but vision needs locality. Flattening each patch gives a vector that attention can mix, while preserving the patch grid keeps the model aware that neighboring tokens were neighboring image regions.

### 2. Window attention cuts the pair count

Full attention compares every token to every other token. On an H×W grid that is `(H·W)^2` query-key pairs, which grows quickly for dense vision. Swin partitions the token grid into small windows and runs attention independently inside each window. A 4×4 token grid has 16 tokens, so global attention uses 16×16 = 256 pairs; with 2×2 windows, there are 4 windows and each uses 4×4 = 16 pairs, for only 64 total.

In [ ]:
grid_ids_w = np.arange(16).reshape(4, 4)  # token ids on a 4x4 patch-token grid.
win_w = 2  # 2x2 local attention windows.
print("token grid:\n", grid_ids_w)
print("global attention pairs:", grid_ids_w.size * grid_ids_w.size)

▶ What you'll see: 16 token ids; global attention would compare all 16 against all 16.

In [ ]:
windows_w = grid_ids_w.reshape(2, win_w, 2, win_w).transpose(0, 2, 1, 3).reshape(-1, win_w * win_w)
local_pairs_w = windows_w.shape[0] * (win_w * win_w) ** 2
print("windows:\n", windows_w)
print("window attention pairs:", local_pairs_w)
print("fraction of global pairs:", round(local_pairs_w / 256, 3))
assert local_pairs_w == 64
assert round(local_pairs_w / 256, 3) == 0.25

▶ What you'll see: four windows, each holding four neighboring tokens; the pair count is one quarter of global attention.

In [ ]:
colors_w = np.zeros_like(grid_ids_w)
for k_w, ids_w in enumerate(windows_w):
    for token_w in ids_w:
        y_w, x_w = divmod(int(token_w), 4)
        colors_w[y_w, x_w] = k_w
plt.figure(figsize=(3.6, 3.2))
plt.imshow(colors_w, cmap="tab10")
for y_w in range(4):
    for x_w in range(4):
        plt.text(x_w, y_w, str(grid_ids_w[y_w, x_w]), ha="center", va="center", color="white")
plt.title("2: independent 2x2 attention windows")
plt.xticks([]); plt.yticks([]); plt.show()

▶ What you'll see: tokens are grouped into four colored local windows; attention is allowed only within each color.

*Why it's done this way:* the computation saved is not a trick after the fact — it is the modeling assumption. Nearby patches usually share edges, textures, and parts, so local attention gets the strongest short-range interactions while keeping cost roughly linear in image size for fixed window size.

### 3. Attention inside a window is ordinary softmax attention

Swin does not invent a new attention formula. Inside each window it still computes `softmax(QKᵀ / sqrt(d) + B)V`, where `B` is a relative-position bias that says nearby offsets inside the window can have learned preferences. The window only changes which tokens are allowed to participate.

In [ ]:
Q_w = np.eye(2)  # two simple query vectors.
K_w = np.eye(2)  # matching key vectors.
V_w = np.array([[2.0, 0.0], [0.0, 3.0]])  # values to be mixed by attention weights.
scores_w = Q_w @ K_w.T  # raw dot-product similarity.
print("scores:\n", scores_w)

▶ What you'll see: each token matches itself with score 1 and the other token with score 0.

In [ ]:
def softmax_rows_w(A_w):
    shifted_w = A_w - A_w.max(axis=1, keepdims=True)
    exp_w = np.exp(shifted_w)
    return exp_w / exp_w.sum(axis=1, keepdims=True)

weights_w = softmax_rows_w(scores_w)
out_w = weights_w @ V_w
print("attention weights:\n", np.round(weights_w, 3))
print("window outputs:\n", np.round(out_w, 3))
assert np.allclose(np.round(weights_w[0], 3), [0.731, 0.269])
assert np.allclose(np.round(out_w[0], 3), [1.462, 0.807])

▶ What you'll see: the first output is mostly value 0, but it still blends in some value 1.

In [ ]:
rel_bias_w = np.array([[0.2, -0.2], [-0.2, 0.2]])  # prefer same-position/self-like relations in this toy window.
biased_weights_w = softmax_rows_w(scores_w + rel_bias_w)
print("biased weights:\n", np.round(biased_weights_w, 3))
plt.figure(figsize=(5, 2.8))
plt.subplot(1, 2, 1); plt.imshow(weights_w, vmin=0, vmax=1, cmap="magma"); plt.title("no bias"); plt.colorbar(fraction=0.046)
plt.subplot(1, 2, 2); plt.imshow(biased_weights_w, vmin=0, vmax=1, cmap="magma"); plt.title("with relative bias"); plt.colorbar(fraction=0.046)
plt.suptitle("3: relative bias changes attention weights")
plt.show()

▶ What you'll see: the biased map puts even more mass on same-position matches.

*Why it's done this way:* softmax turns similarity scores into a normalized weighted average, so the output remains a mixture of values. Adding relative position bias before the softmax changes the probabilities, not the values, which is exactly where spatial preference belongs: it says which neighbor should be listened to more.

### 4. Shifted windows let information cross old borders

If Swin only used fixed windows, tokens near a window border would never talk to tokens just across that border. The shifted-window block repairs this by rolling the grid before partitioning, so the next set of windows straddles old boundaries. In a 1-D row of eight tokens with window size 4 and shift 2, the shifted grouping includes `[2, 3, 4, 5]`, so tokens 3 and 4 finally share attention.

In [ ]:
row_w = np.arange(8)  # one row of token ids.
first_windows_w = row_w.reshape(2, 4)  # unshifted windows of length 4.
shift_w = 2
rolled_w = np.roll(row_w, -shift_w)  # shift left before grouping.
print("unshifted windows:\n", first_windows_w)
print("rolled row:", rolled_w)

▶ What you'll see: tokens 0–3 and 4–7 are separate before the shift.

In [ ]:
shift_windows_w = rolled_w.reshape(2, 4)
print("shifted grouping before unroll:\n", shift_windows_w)
print("middle cross-border group:", np.array([2, 3, 4, 5]))
assert 3 in np.array([2, 3, 4, 5]) and 4 in np.array([2, 3, 4, 5])

▶ What you'll see: a shifted group crosses the old boundary between tokens 3 and 4.

In [ ]:
pos_w = np.arange(8)
plt.figure(figsize=(6, 2.4))
plt.scatter(pos_w, np.zeros_like(pos_w), c=[0,0,0,0,1,1,1,1], cmap="tab10", s=120)
plt.scatter(pos_w, np.ones_like(pos_w), c=[1,1,0,0,0,0,1,1], cmap="tab10", s=120)
for i_w in pos_w:
    plt.text(i_w, -0.08, str(i_w), ha="center")
    plt.text(i_w, 0.92, str(i_w), ha="center")
plt.yticks([0, 1], ["fixed", "shifted"]); plt.xticks([])
plt.title("4: shifted windows change who can attend together")
plt.show()

▶ What you'll see: the color boundary moves, so former cross-border neighbors become same-window neighbors in the shifted row.

*Why it's done this way:* stacking local blocks expands the receptive field only if the local neighborhoods change. The shift is a cheap deterministic way to rewire neighborhoods without paying global-attention cost at every layer.

### 5. Boundary masks prevent wraparound mistakes

A cyclic roll is convenient for code, but it creates a fake adjacency: tokens from the right edge can wrap next to tokens from the left edge. Real Swin masks those wrapped pairs inside shifted windows so opposite image edges do not attend as if they were neighbors. The mask is not optional bookkeeping; it preserves the geometry of the original image.

In [ ]:
labels_w = np.repeat(np.arange(2), 4)  # original window id before rolling: 0 for left, 1 for right.
rolled_labels_w = np.roll(labels_w, -shift_w)
rolled_tokens_w = np.roll(row_w, -shift_w)
print("rolled tokens:", rolled_tokens_w)
print("rolled original-window labels:", rolled_labels_w)

▶ What you'll see: after rolling, the last shifted window mixes labels from the original right and left edges.

In [ ]:
last_tokens_w = rolled_tokens_w.reshape(2, 4)[1]
last_labels_w = rolled_labels_w.reshape(2, 4)[1]
mask_w = last_labels_w[:, None] == last_labels_w[None, :]  # True pairs are allowed; False pairs crossed the wrap seam.
print("last shifted window tokens:", last_tokens_w)
print("allowed-pair mask:\n", mask_w.astype(int))
assert mask_w[0, 2] == False

▶ What you'll see: some token pairs in the wrapped window are masked out because they came from different original edge regions.

In [ ]:
plt.figure(figsize=(3.6, 3.2))
plt.imshow(mask_w.astype(int), cmap="Greys", vmin=0, vmax=1)
plt.xticks(range(4), last_tokens_w); plt.yticks(range(4), last_tokens_w)
plt.title("5: shifted-window boundary mask")
plt.xlabel("key token"); plt.ylabel("query token")
plt.show()

▶ What you'll see: white/allowed blocks are separated from black/disallowed wraparound pairs.

*Why it's done this way:* rolling lets every shifted window remain the same size for vectorized computation, but the mask restores the non-cyclic image boundary. Without it, the model would learn from impossible neighbors across opposite edges.

### 6. Patch merging builds the hierarchy

Dense vision tasks need multi-scale features: early layers should keep local detail, while later layers should see larger objects. Swin gets this by merging each 2×2 group of tokens. If the incoming grid is 4×4 with C=3 channels, merging halves each spatial side to 2×2, concatenates four tokens into 4C=12 channels, then projects to a wider feature such as 2C=6 channels.

In [ ]:
X_w = np.arange(4 * 4 * 3).reshape(4, 4, 3).astype(float)  # 4x4 token grid with C=3 channels.
merged_blocks_w = X_w.reshape(2, 2, 2, 2, 3).transpose(0, 2, 1, 3, 4).reshape(2, 2, 12)
print("before:", X_w.shape)
print("after concatenating 2x2 neighborhoods:", merged_blocks_w.shape)
assert merged_blocks_w.shape == (2, 2, 12)

▶ What you'll see: the spatial grid becomes 2×2, while channels temporarily become 12.

In [ ]:
proj_w = np.linspace(-0.1, 0.1, 12 * 6).reshape(12, 6)  # a tiny deterministic projection from 4C to 2C.
merged_w = merged_blocks_w @ proj_w
print("projected merged shape:", merged_w.shape)
print("one merged token:", np.round(merged_w[0, 0], 3))
assert merged_w.shape == (2, 2, 6)

▶ What you'll see: each merged token now has 6 channels: coarser spatially, richer in features.

In [ ]:
plt.figure(figsize=(5, 2.8))
plt.subplot(1, 2, 1); plt.imshow(X_w[:, :, 0], cmap="viridis"); plt.title("4x4 stage")
plt.subplot(1, 2, 2); plt.imshow(merged_w[:, :, 0], cmap="viridis"); plt.title("2x2 merged stage")
plt.suptitle("6: patch merging creates a feature pyramid")
plt.show()

▶ What you'll see: the feature map is lower-resolution after merging, like a CNN pyramid stage.

*Why it's done this way:* concatenating a 2×2 neighborhood preserves the local evidence before compression, and projecting to more channels lets the coarser token carry richer information. This is why Swin can serve as a backbone for detectors and segmenters that expect multi-scale maps.

## 🛠️ Setup

In [ ]:
import numpy as np # Load NumPy for arrays, reshaping, masks, softmax, and small numerical checks.
import matplotlib.pyplot as plt # Load Matplotlib for the heatmaps, bars, scatters, and line plots used in this lesson.
np.random.seed(0) # Fix the global random seed so every stochastic example is repeatable.

## 🟢 Basics (warm-up)

### Basic 1 — Split an image into patch tokens

**Goal.** Turn a tiny image into non-overlapping patch tokens, because Swin begins with the same patch-token representation as ViT. We build it in 2 steps.

In [ ]:
img_b1 = np.arange(16).reshape(4, 4) # Create a 4x4 toy image whose pixel values identify positions.
patch_b1 = 2 # Choose 2x2 patches so exactly four patches fit in the image.
print("image shape:", img_b1.shape, "patch size:", patch_b1) # Inspect the ingredients before reshaping.
print(img_b1) # Print the image so each patch can be checked by eye.

▶ What you'll see: a 4×4 grid of numbers from 0 to 15.

In [ ]:
patches_b1 = img_b1.reshape(2, patch_b1, 2, patch_b1).transpose(0, 2, 1, 3) # Arrange as patch-row, patch-col, pixel-row, pixel-col.
tokens_b1 = patches_b1.reshape(4, patch_b1 * patch_b1) # Flatten each 2x2 patch into one token row.
print("tokens:\n", tokens_b1) # Inspect the four flattened patch tokens.
assert tokens_b1.shape == (4, 4) # Verify four patches and four pixels per patch.
assert np.all(tokens_b1[0] == np.array([0, 1, 4, 5])) # Verify the top-left patch token.

▶ What you'll see: a 4×4 token table, not a neural network yet — just patch extraction.

👀 Takeaway: Swin starts from ordinary patch tokens, then changes how those tokens are mixed.

### Basic 2 — Keep tokens as a 2-D grid

**Goal.** Store patch tokens with row and column positions, because windows and shifts are spatial operations. We build it in 2 steps.

In [ ]:
tokens_b2 = np.arange(2 * 2 * 4).reshape(2, 2, 4) # Create a 2x2 grid of four-dimensional patch tokens.
print("token-grid shape:", tokens_b2.shape) # Inspect grid height, grid width, and channel dimension.
print("token at row 1, col 0:", tokens_b2[1, 0]) # Read one token by spatial address.

▶ What you'll see: tokens have both a grid location and a feature vector.

In [ ]:
plt.figure(figsize=(3.5, 3)) # Start a compact grid visualization.
plt.imshow(np.arange(4).reshape(2, 2), cmap="tab10") # Color the four token positions.
for y_b2 in range(2): # Loop over token rows.
    for x_b2 in range(2): # Loop over token columns.
        plt.text(x_b2, y_b2, f"({y_b2},{x_b2})", ha="center", va="center", color="white") # Label each token address.
plt.title("Basic 2: patch-token grid") # Title the spatial layout plot.
plt.xticks([]); plt.yticks([]); plt.show() # Hide axes and display the grid.

▶ What you'll see: four tokens arranged as an image grid rather than an anonymous sequence.

👀 Takeaway: Swin's key operations require token positions, so the patch grid matters.

### Basic 3 — Count global attention pairs

**Goal.** Count query-key comparisons for global attention, because Swin's efficiency claim starts with this cost. We build it in 2 steps.

In [ ]:
H_b3 = 4 # Use a 4-token height.
W_b3 = 4 # Use a 4-token width.
n_tokens_b3 = H_b3 * W_b3 # Count total patch tokens.
print("tokens:", n_tokens_b3) # Inspect how many queries and keys global attention has.

▶ What you'll see: a 4×4 grid contains 16 tokens.

In [ ]:
global_pairs_b3 = n_tokens_b3 * n_tokens_b3 # Every query compares with every key.
print("global attention pairs:", global_pairs_b3) # Inspect the quadratic pair count.
assert global_pairs_b3 == 256 # Verify 16 times 16.
plt.figure(figsize=(4, 3)) # Create a compact cost chart.
plt.bar(["global pairs"], [global_pairs_b3], color="crimson") # Show the full-attention cost for this grid.
plt.title("Basic 3: global attention cost") # Title the cost plot.
plt.ylabel("query-key pairs") # Label the pair-count axis.
plt.show() # Display the plot.

▶ What you'll see: even a tiny 16-token grid already has 256 attention comparisons.

👀 Takeaway: global attention cost grows quadratically with the number of image patches.

### Basic 4 — Partition a grid into windows

**Goal.** Group neighboring tokens into fixed 2×2 windows, because window attention mixes only local neighborhoods. We build it in 2 steps.

In [ ]:
grid_b4 = np.arange(16).reshape(4, 4) # Create token ids on a 4x4 grid.
win_b4 = 2 # Choose 2x2 local windows.
print("grid:\n", grid_b4) # Inspect token positions before partitioning.

▶ What you'll see: token ids arranged in their spatial order.

In [ ]:
windows_b4 = grid_b4.reshape(2, win_b4, 2, win_b4).transpose(0, 2, 1, 3).reshape(-1, win_b4 * win_b4) # Partition into non-overlapping windows.
print("windows:\n", windows_b4) # Inspect each local attention group.
assert windows_b4.shape == (4, 4) # Verify four windows with four tokens each.
assert np.all(windows_b4[0] == np.array([0, 1, 4, 5])) # Verify the top-left window members.

▶ What you'll see: each row lists the four token ids allowed to attend together.

👀 Takeaway: window attention replaces one large all-to-all group with many small local groups.

### Basic 5 — Compare global and window pair counts

**Goal.** Compute the savings from window attention, because Swin scales by reducing attention pairs. We build it in 3 steps.

In [ ]:
n_tokens_b5 = 16 # Use the same 4x4 grid as the previous examples.
win_tokens_b5 = 4 # A 2x2 window contains four tokens.
n_windows_b5 = n_tokens_b5 // win_tokens_b5 # Count how many non-overlapping windows tile the grid.
print("windows:", n_windows_b5, "tokens per window:", win_tokens_b5) # Inspect the tiling numbers.

▶ What you'll see: the grid becomes four local windows.

In [ ]:
global_pairs_b5 = n_tokens_b5 ** 2 # Count global query-key comparisons.
window_pairs_b5 = n_windows_b5 * win_tokens_b5 ** 2 # Count local comparisons inside all windows.
ratio_b5 = window_pairs_b5 / global_pairs_b5 # Compute the fraction of global work used by windows.
print("global:", global_pairs_b5, "window:", window_pairs_b5, "ratio:", ratio_b5) # Inspect the savings.
assert window_pairs_b5 == 64 # Verify the lesson count.
assert ratio_b5 == 0.25 # Verify the local attention uses one quarter of the pairs here.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact comparison plot.
plt.bar(["global", "window"], [global_pairs_b5, window_pairs_b5], color=["crimson", "seagreen"]) # Compare pair counts.
plt.title("Basic 5: attention pair savings") # Title the cost comparison.
plt.ylabel("query-key pairs") # Label the cost scale.
plt.show() # Display the bar chart.

▶ What you'll see: the window bar is much shorter than the global bar.

👀 Takeaway: fixed-size windows make attention cost proportional to the number of windows rather than all token pairs.

### Basic 6 — Compute row softmax attention

**Goal.** Convert similarity scores into attention weights, because Swin uses the same softmax weighting inside each window as a transformer. We build it in 3 steps.

In [ ]:
scores_b6 = np.array([[1.0, 0.0], [0.0, 1.0]]) # Define two-token self-similarity scores.
values_b6 = np.array([[2.0, 0.0], [0.0, 3.0]]) # Define value vectors to be mixed.
print("scores:\n", scores_b6) # Inspect raw attention scores.

▶ What you'll see: each token has a higher score with itself than with the other token.

In [ ]:
exp_b6 = np.exp(scores_b6 - scores_b6.max(axis=1, keepdims=True)) # Stabilize and exponentiate each row.
weights_b6 = exp_b6 / exp_b6.sum(axis=1, keepdims=True) # Normalize each row into probabilities.
print("weights:\n", np.round(weights_b6, 3)) # Inspect softmax attention weights.
assert np.allclose(np.round(weights_b6[0], 3), [0.731, 0.269]) # Verify the canonical softmax row.

In [ ]:
out_b6 = weights_b6 @ values_b6 # Mix value vectors by the attention weights.
print("attention output:\n", np.round(out_b6, 3)) # Inspect the weighted value outputs.
plt.figure(figsize=(4, 3)) # Create a compact heatmap.
plt.imshow(weights_b6, cmap="magma", vmin=0, vmax=1) # Visualize attention probabilities.
plt.colorbar(label="weight") # Add a colorbar for probability values.
plt.title("Basic 6: softmax attention weights") # Title the attention map.
plt.show() # Display the heatmap.

▶ What you'll see: each output mostly copies its own value but blends in the other token.

👀 Takeaway: Swin changes the attention neighborhood, not the softmax attention rule inside it.

### Basic 7 — Add relative position bias

**Goal.** Bias attention by relative location inside a window, because spatial order still matters after patching. We build it in 3 steps.

In [ ]:
scores_b7 = np.zeros((4, 4)) # Start with equal content similarity among four window tokens.
coords_b7 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) # Store 2x2 window coordinates for each token.
print("coords:\n", coords_b7) # Inspect token positions inside the window.

▶ What you'll see: four tokens have explicit row-column coordinates.

In [ ]:
dist_b7 = np.abs(coords_b7[:, None, :] - coords_b7[None, :, :]).sum(axis=2) # Manhattan distance between token positions.
bias_b7 = -0.5 * dist_b7 # Prefer nearby tokens by giving farther tokens lower logits.
print("relative-position bias:\n", bias_b7) # Inspect the spatial bias matrix.
assert bias_b7[0, 3] == -1.0 # Diagonal-opposite tokens are distance 2, so the bias is -1.

In [ ]:
exp_b7 = np.exp(scores_b7 + bias_b7 - (scores_b7 + bias_b7).max(axis=1, keepdims=True)) # Exponentiate biased logits.
weights_b7 = exp_b7 / exp_b7.sum(axis=1, keepdims=True) # Normalize into attention probabilities.
print("first query weights:", np.round(weights_b7[0], 3)) # Inspect how position shapes attention.
plt.figure(figsize=(4, 3)) # Create a compact heatmap.
plt.imshow(weights_b7, cmap="magma", vmin=0, vmax=1) # Visualize biased attention weights.
plt.colorbar(label="weight") # Add a probability scale.
plt.title("Basic 7: relative position bias") # Title the attention map.
plt.show() # Display the heatmap.

▶ What you'll see: a token pays most attention to itself, less to adjacent tokens, and least to the diagonal opposite token.

👀 Takeaway: relative bias gives local attention a learnable sense of spatial arrangement.

### Basic 8 — Shift a token grid

**Goal.** Roll the grid before partitioning, because shifted windows let different neighbors communicate in alternating blocks. We build it in 2 steps.

In [ ]:
grid_b8 = np.arange(16).reshape(4, 4) # Create token ids on a 4x4 grid.
shift_b8 = 1 # Shift by one token along both spatial axes for this small demo.
shifted_b8 = np.roll(np.roll(grid_b8, -shift_b8, axis=0), -shift_b8, axis=1) # Roll up and left.
print("original:\n", grid_b8) # Inspect original token positions.
print("shifted:\n", shifted_b8) # Inspect shifted positions before windowing.

▶ What you'll see: token ids move up-left, with edge tokens wrapping around.

In [ ]:
plt.figure(figsize=(5, 2.6)) # Create a side-by-side visualization.
plt.subplot(1, 2, 1); plt.imshow(grid_b8, cmap="viridis"); plt.title("original") # Show original grid.
plt.subplot(1, 2, 2); plt.imshow(shifted_b8, cmap="viridis"); plt.title("shifted") # Show shifted grid.
plt.suptitle("Basic 8: cyclic shift before windows") # Add an overall title.
plt.show() # Display both grids.

▶ What you'll see: the partition will now cut the image at different token boundaries.

👀 Takeaway: shifting changes local neighborhoods without using global attention.

### Basic 9 — Mask wrapped boundary pairs

**Goal.** Identify fake wraparound neighbors after a cyclic shift, because opposite image edges should not attend as if adjacent. We build it in 3 steps.

In [ ]:
labels_b9 = np.repeat(np.arange(2), 4) # Original 1-D window labels for eight tokens.
tokens_b9 = np.arange(8) # Token ids in a 1-D toy row.
rolled_tokens_b9 = np.roll(tokens_b9, -2) # Shift left by two positions.
rolled_labels_b9 = np.roll(labels_b9, -2) # Shift the original labels the same way.
print("rolled tokens:", rolled_tokens_b9) # Inspect shifted token order.
print("rolled labels:", rolled_labels_b9) # Inspect original-region labels after shift.

▶ What you'll see: shifted windows can contain tokens from different original edge regions.

In [ ]:
window_tokens_b9 = rolled_tokens_b9.reshape(2, 4)[1] # Select the wrapped window.
window_labels_b9 = rolled_labels_b9.reshape(2, 4)[1] # Select labels for the same window.
mask_b9 = window_labels_b9[:, None] == window_labels_b9[None, :] # Allow only pairs from the same original region.
print("window tokens:", window_tokens_b9) # Inspect token ids in the wrapped window.
print("mask:\n", mask_b9.astype(int)) # Inspect allowed and blocked pairs.
assert mask_b9[0, 2] == False # Verify a wraparound pair is blocked.

In [ ]:
plt.figure(figsize=(3.8, 3.2)) # Create a compact mask heatmap.
plt.imshow(mask_b9.astype(int), cmap="Greys", vmin=0, vmax=1) # Visualize allowed pairs as white.
plt.xticks(range(4), window_tokens_b9); plt.yticks(range(4), window_tokens_b9) # Label rows and columns by token ids.
plt.title("Basic 9: shifted-window mask") # Title the boundary mask.
plt.show() # Display the mask.

▶ What you'll see: the mask blocks attention across the artificial wrap seam.

👀 Takeaway: shifted windows need masking so implementation convenience does not corrupt image geometry.

### Basic 10 — Merge 2×2 patches

**Goal.** Halve spatial resolution and increase channels, because Swin builds a hierarchy like a vision backbone. We build it in 3 steps.

In [ ]:
X_b10 = np.arange(4 * 4 * 3).reshape(4, 4, 3).astype(float) # Create a 4x4 token grid with three channels.
print("input shape:", X_b10.shape) # Inspect the stage resolution and channel count.

▶ What you'll see: a feature map with height 4, width 4, and C=3 channels.

In [ ]:
concat_b10 = X_b10.reshape(2, 2, 2, 2, 3).transpose(0, 2, 1, 3, 4).reshape(2, 2, 12) # Concatenate each 2x2 neighborhood.
print("after 2x2 concat:", concat_b10.shape) # Inspect the halved grid and 4C channels.
assert concat_b10.shape == (2, 2, 12) # Verify 4C = 12 channels before projection.

In [ ]:
W_b10 = np.ones((12, 6)) / 12 # Use a simple averaging-style projection from 4C to 2C.
merged_b10 = concat_b10 @ W_b10 # Project concatenated neighborhoods to six channels.
print("merged shape:", merged_b10.shape) # Inspect the final hierarchy stage shape.
assert merged_b10.shape == (2, 2, 6) # Verify the stage is 2x2 with 2C channels.
plt.figure(figsize=(4, 3)) # Create a compact heatmap.
plt.imshow(merged_b10[:, :, 0], cmap="viridis") # Visualize one merged channel.
plt.title("Basic 10: patch-merged feature map") # Title the merged feature map.
plt.colorbar(label="channel 0") # Add a value scale.
plt.show() # Display the heatmap.

▶ What you'll see: a 2×2 map, showing that the stage became coarser but not empty.

👀 Takeaway: patch merging is Swin's pyramid mechanism: lower resolution, more channels.

## 🟡 Easy

### Easy 1 — Run window attention on a 4×4 grid

**Goal.** Apply attention independently inside each 2×2 window, because W-MSA mixes local tokens without crossing window boundaries. We build it in 4 steps.

In [ ]:
X_e1 = np.arange(16, dtype=float).reshape(4, 4, 1) / 10 # Create a 4x4 token grid with one feature channel.
win_e1 = 2 # Use 2x2 attention windows.
print("input feature map:\n", np.round(X_e1[:, :, 0], 2)) # Inspect local scalar features.

▶ What you'll see: token values increase from top-left to bottom-right.

In [ ]:
windows_e1 = X_e1.reshape(2, win_e1, 2, win_e1, 1).transpose(0, 2, 1, 3, 4).reshape(-1, 4, 1) # Partition into four windows.
print("window tensor shape:", windows_e1.shape) # Inspect number of windows, tokens per window, and channels.
assert windows_e1.shape == (4, 4, 1) # Verify four local groups.

In [ ]:
out_windows_e1 = [] # Store one attended output per window.
for W_e1 in windows_e1: # Loop over local windows.
    scores_e1 = W_e1 @ W_e1.T # Use scalar dot products as attention logits.
    exp_e1 = np.exp(scores_e1 - scores_e1.max(axis=1, keepdims=True)) # Stabilize exponentials.
    attn_e1 = exp_e1 / exp_e1.sum(axis=1, keepdims=True) # Normalize rows into probabilities.
    out_windows_e1.append(attn_e1 @ W_e1) # Mix values inside this window only.
out_windows_e1 = np.array(out_windows_e1) # Stack attended windows.
print("first window output:", np.round(out_windows_e1[0, :, 0], 3)) # Inspect local mixing for window 0.

In [ ]:
out_e1 = out_windows_e1.reshape(2, 2, 2, 2, 1).transpose(0, 2, 1, 3, 4).reshape(4, 4, 1) # Reverse the partition to the image grid.
plt.figure(figsize=(5, 2.8)) # Create a before-after plot.
plt.subplot(1, 2, 1); plt.imshow(X_e1[:, :, 0], cmap="viridis"); plt.title("input") # Show original features.
plt.subplot(1, 2, 2); plt.imshow(out_e1[:, :, 0], cmap="viridis"); plt.title("W-MSA output") # Show locally mixed features.
plt.suptitle("Easy 1: attention inside each 2x2 window") # Add an overall title.
plt.show() # Display the comparison.

▶ What you'll see: each 2×2 block is smoothed/mixed internally, with no information crossing block boundaries.

👀 Takeaway: W-MSA is ordinary attention repeated independently over local windows.

### Easy 2 — Show shifted windows crossing a boundary

**Goal.** Compare fixed and shifted window memberships, because alternating partitions are how Swin passes information across old borders. We build it in 3 steps.

In [ ]:
grid_e2 = np.arange(16).reshape(4, 4) # Create token ids on a 4x4 grid.
win_e2 = 2 # Use 2x2 windows.
fixed_e2 = grid_e2.reshape(2, win_e2, 2, win_e2).transpose(0, 2, 1, 3).reshape(-1, 4) # Fixed windows.
print("fixed windows:\n", fixed_e2) # Inspect first-block memberships.

▶ What you'll see: token 5 attends with 4, 1, and 0 in the fixed partition.

In [ ]:
shifted_grid_e2 = np.roll(np.roll(grid_e2, -1, axis=0), -1, axis=1) # Shift by one token in both directions.
shifted_e2 = shifted_grid_e2.reshape(2, win_e2, 2, win_e2).transpose(0, 2, 1, 3).reshape(-1, 4) # Partition after shifting.
print("shifted windows before unroll:\n", shifted_e2) # Inspect changed memberships.
contains_5_10_e2 = any((5 in row_e2 and 10 in row_e2) for row_e2 in shifted_e2) # Check a diagonal cross-boundary pair.
print("tokens 5 and 10 share a shifted window?", contains_5_10_e2) # Inspect the boundary-crossing result.
assert contains_5_10_e2 == True # Verify shifted partition connects a new pair.

In [ ]:
plt.figure(figsize=(5, 2.6)) # Create a visual comparison of partitions.
plt.subplot(1, 2, 1); plt.imshow(grid_e2 // 2 + (grid_e2 % 2 == 0), cmap="tab10"); plt.title("fixed ids") # Show fixed-like pattern.
plt.subplot(1, 2, 2); plt.imshow(shifted_grid_e2 // 2 + (shifted_grid_e2 % 2 == 0), cmap="tab10"); plt.title("shifted ids") # Show shifted token positions.
plt.suptitle("Easy 2: shifting changes window membership") # Title the comparison.
plt.show() # Display the grids.

▶ What you'll see: after shifting, tokens that were separated by a fixed window boundary can land in the same local group.

👀 Takeaway: shifted windows are the communication bridge between local regions.

### Easy 3 — Build a relative-bias table for a 2×2 window

**Goal.** Convert relative offsets into a bias matrix, because Swin learns attention preferences by displacement inside a window. We build it in 4 steps.

In [ ]:
coords_e3 = np.array([[0, 0], [0, 1], [1, 0], [1, 1]]) # Token coordinates in a 2x2 window.
offsets_e3 = coords_e3[:, None, :] - coords_e3[None, :, :] # Query-key relative offsets.
print("offsets shape:", offsets_e3.shape) # Inspect query, key, and dy/dx dimensions.

▶ What you'll see: every query-key pair has a 2-D relative displacement.

In [ ]:
unique_offsets_e3 = sorted({tuple(o_e3) for row_e3 in offsets_e3 for o_e3 in row_e3}) # List possible offsets.
bias_values_e3 = {off_e3: -0.25 * (abs(off_e3[0]) + abs(off_e3[1])) for off_e3 in unique_offsets_e3} # Penalize farther offsets in this toy table.
print("bias table:", bias_values_e3) # Inspect the offset-to-bias mapping.
assert bias_values_e3[(0, 0)] == 0.0 # Same-position offset has no penalty.

In [ ]:
bias_e3 = np.array([[bias_values_e3[tuple(offsets_e3[i_e3, j_e3])] for j_e3 in range(4)] for i_e3 in range(4)]) # Materialize pairwise bias matrix.
print("bias matrix:\n", bias_e3) # Inspect the logits added to attention scores.
assert bias_e3[0, 3] == -0.5 # Diagonal-opposite tokens are two steps apart.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact bias heatmap.
plt.imshow(bias_e3, cmap="coolwarm", vmin=-0.5, vmax=0) # Visualize more negative values for farther offsets.
plt.colorbar(label="bias logit") # Add a bias-value scale.
plt.title("Easy 3: relative-position bias matrix") # Title the bias plot.
plt.xlabel("key token"); plt.ylabel("query token") # Label attention axes.
plt.show() # Display the heatmap.

▶ What you'll see: self positions have the largest bias, adjacent positions are lower, and diagonal positions are lowest.

👀 Takeaway: relative bias injects spatial geometry directly into attention logits.

### Easy 4 — Patch merge with a learned-style projection

**Goal.** Concatenate 2×2 neighborhoods and project channels, because Swin stages become coarser and wider. We build it in 4 steps.

In [ ]:
X_e4 = np.arange(4 * 4 * 2).reshape(4, 4, 2).astype(float) # Create a 4x4 grid with C=2 channels.
print("input shape:", X_e4.shape) # Inspect the starting stage.

▶ What you'll see: the feature map begins at 4×4 resolution with two channels.

In [ ]:
concat_e4 = X_e4.reshape(2, 2, 2, 2, 2).transpose(0, 2, 1, 3, 4).reshape(2, 2, 8) # Concatenate four neighboring tokens.
print("concat shape:", concat_e4.shape) # Inspect 4C channels.
assert concat_e4.shape[-1] == 8 # Verify 4 times C.

In [ ]:
proj_e4 = np.arange(8 * 4).reshape(8, 4) / 100 # Create a deterministic 4C-to-2C projection matrix.
Y_e4 = concat_e4 @ proj_e4 # Project each merged neighborhood to 2C channels.
print("merged shape:", Y_e4.shape) # Inspect output resolution and channels.
assert Y_e4.shape == (2, 2, 4) # Verify H/2, W/2, 2C.

In [ ]:
plt.figure(figsize=(5, 2.8)) # Create a before-after resolution plot.
plt.subplot(1, 2, 1); plt.imshow(X_e4[:, :, 0], cmap="viridis"); plt.title("4x4 input") # Show one input channel.
plt.subplot(1, 2, 2); plt.imshow(Y_e4[:, :, 0], cmap="viridis"); plt.title("2x2 merged") # Show one merged channel.
plt.suptitle("Easy 4: patch merging halves resolution") # Add overall title.
plt.show() # Display both stages.

▶ What you'll see: a smaller 2×2 feature map with information derived from each 2×2 input block.

👀 Takeaway: patch merging trades spatial detail for broader, richer token features.

### Easy 5 — Track stage shapes through a tiny Swin backbone

**Goal.** Compute the grid and channel sizes after repeated patch merging, because Swin is a hierarchical backbone. We build it in 3 steps.

In [ ]:
H_e5, W_e5, C_e5 = 8, 8, 3 # Start from an 8x8 patch-token grid with three channels.
stages_e5 = [(H_e5, W_e5, C_e5)] # Store the first stage shape.
print("stage 0:", stages_e5[0]) # Inspect the initial stage.

▶ What you'll see: the toy backbone starts with an 8×8 grid.

In [ ]:
for _e5 in range(3): # Apply three patch-merging transitions.
    H_e5, W_e5, C_e5 = H_e5 // 2, W_e5 // 2, C_e5 * 2 # Halve resolution and double channels.
    stages_e5.append((H_e5, W_e5, C_e5)) # Store the new stage shape.
print("stages:", stages_e5) # Inspect the whole pyramid.
assert stages_e5[-1] == (1, 1, 24) # Verify the final tiny stage.

In [ ]:
areas_e5 = np.array([h_e5 * w_e5 for h_e5, w_e5, c_e5 in stages_e5]) # Count spatial tokens per stage.
channels_e5 = np.array([c_e5 for h_e5, w_e5, c_e5 in stages_e5]) # Collect channel widths per stage.
plt.figure(figsize=(5, 3)) # Create a compact stage plot.
plt.plot(areas_e5, marker="o", label="tokens") # Plot token count dropping.
plt.plot(channels_e5, marker="s", label="channels") # Plot channel count rising.
plt.title("Easy 5: Swin pyramid shapes") # Title the stage-shape plot.
plt.xlabel("stage") # Label the stage axis.
plt.legend() # Show curve labels.
plt.show() # Display the plot.

▶ What you'll see: token count falls while channel count rises across stages.

👀 Takeaway: Swin restores the multi-scale pyramid structure that dense vision systems expect.

## 🔴 Advanced

### Advanced 1 — Compare pair counts as image size grows

**Goal.** Sweep grid sizes and compare global versus window attention costs, because Swin's advantage increases with high-resolution feature maps. We build it in 4 steps.

In [ ]:
sizes_a1 = np.array([8, 16, 32, 64]) # Test square token grids of increasing width.
window_a1 = 4 # Use fixed 4x4 attention windows.
print("grid sizes:", sizes_a1) # Inspect the resolution sweep.

▶ What you'll see: the experiment covers small through moderately large token grids.

In [ ]:
tokens_a1 = sizes_a1 ** 2 # Count tokens per square grid.
global_pairs_a1 = tokens_a1 ** 2 # Global attention compares every token to every token.
window_pairs_a1 = (tokens_a1 // (window_a1 ** 2)) * (window_a1 ** 2) ** 2 # Window attention uses fixed-size local groups.
print("global pairs:", global_pairs_a1) # Inspect quadratic costs.
print("window pairs:", window_pairs_a1) # Inspect local-window costs.
assert window_pairs_a1[0] == 1024 # 4 windows times 16x16 pairs for an 8x8 grid.

In [ ]:
ratio_a1 = window_pairs_a1 / global_pairs_a1 # Compute fraction of global comparisons.
print("window/global ratios:", np.round(ratio_a1, 4)) # Inspect relative cost as size grows.
assert ratio_a1[-1] < 0.004 # Verify the high-resolution relative cost is tiny.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a log-scale cost plot.
plt.plot(sizes_a1, global_pairs_a1, marker="o", label="global") # Plot global pair count.
plt.plot(sizes_a1, window_pairs_a1, marker="s", label="window") # Plot window pair count.
plt.yscale("log") # Use log scale so both curves are visible.
plt.title("Advanced 1: attention pair scaling") # Title the scaling comparison.
plt.xlabel("grid side length") # Label image-token resolution.
plt.ylabel("query-key pairs, log scale") # Label cost axis.
plt.legend() # Show curve labels.
plt.show() # Display the plot.

▶ What you'll see: global cost curves upward much faster than fixed-window cost.

👀 Takeaway: local windows are what let Swin operate on dense, high-resolution feature maps.

### Advanced 2 — Propagate information with alternating windows

**Goal.** Track which tokens can influence a target token after fixed and shifted blocks, because Swin expands context gradually over depth. We build it in 4 steps.

In [ ]:
n_a2 = 8 # Use a one-dimensional row of eight tokens.
win_a2 = 4 # Use windows of length four.
reach_a2 = np.eye(n_a2, dtype=bool) # reach[i,j] says token i can contain information from token j.
print("initial reach for token 3:", np.where(reach_a2[3])[0]) # Inspect that each token initially knows only itself.

▶ What you'll see: token 3 starts with access only to token 3.

In [ ]:
fixed_groups_a2 = [np.array([0, 1, 2, 3]), np.array([4, 5, 6, 7])] # Fixed local windows.
for group_a2 in fixed_groups_a2: # Apply one fixed-window block.
    union_a2 = reach_a2[group_a2].any(axis=0) # Gather information available inside the group.
    reach_a2[group_a2] = union_a2 # Every token in the group receives the same group information.
print("after fixed block, token 3 reaches:", np.where(reach_a2[3])[0]) # Inspect local context.
assert np.all(np.where(reach_a2[3])[0] == np.array([0, 1, 2, 3])) # Verify token 3 knows its fixed window.

In [ ]:
shift_groups_a2 = [np.array([2, 3, 4, 5]), np.array([6, 7, 0, 1])] # Shifted windows after a half-window shift.
for group_a2 in shift_groups_a2: # Apply one shifted-window block.
    union_a2 = reach_a2[group_a2].any(axis=0) # Combine information within the shifted group.
    reach_a2[group_a2] = union_a2 # Broadcast it to tokens in that shifted group.
print("after shifted block, token 3 reaches:", np.where(reach_a2[3])[0]) # Inspect expanded context.
assert 4 in np.where(reach_a2[3])[0] # Verify information crossed the old boundary.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a reachability heatmap.
plt.imshow(reach_a2.astype(int), cmap="Greys", vmin=0, vmax=1) # Show which source tokens can affect each target.
plt.title("Advanced 2: reachability after fixed + shifted blocks") # Title the information-flow plot.
plt.xlabel("source token") # Label columns as information sources.
plt.ylabel("target token") # Label rows as updated tokens.
plt.show() # Display the heatmap.

▶ What you'll see: the reachability row for token 3 expands beyond its original fixed window.

👀 Takeaway: shifted windows turn local attention into progressively larger context over multiple blocks.

### Advanced 3 — Apply a shifted-window attention mask

**Goal.** Add a large negative mask to attention logits, because softmax should assign essentially zero probability to wrapped boundary pairs. We build it in 4 steps.

In [ ]:
tokens_a3 = np.array([2, 3, 4, 5, 6, 7, 0, 1]) # A shifted row where the second window wraps across the image edge.
labels_a3 = np.array([0, 0, 1, 1, 1, 1, 0, 0]) # Original region labels after rolling.
window_tokens_a3 = tokens_a3.reshape(2, 4)[1] # Select the wrapped shifted window.
window_labels_a3 = labels_a3.reshape(2, 4)[1] # Select its original-region labels.
print("wrapped window tokens:", window_tokens_a3) # Inspect the problematic window.

▶ What you'll see: the wrapped window contains tokens from the beginning of the original image row.

In [ ]:
allowed_a3 = window_labels_a3[:, None] == window_labels_a3[None, :] # Allow only same-region pairs.
logits_a3 = np.ones((4, 4)) # Pretend content similarity is equal for every pair.
masked_logits_a3 = np.where(allowed_a3, logits_a3, -100.0) # Replace disallowed logits with a huge negative value.
print("allowed mask:\n", allowed_a3.astype(int)) # Inspect valid attention pairs.
assert masked_logits_a3[0, 2] == -100.0 # Verify a wrapped pair is suppressed.

In [ ]:
exp_a3 = np.exp(masked_logits_a3 - masked_logits_a3.max(axis=1, keepdims=True)) # Stabilize exponentials after masking.
weights_a3 = exp_a3 / exp_a3.sum(axis=1, keepdims=True) # Normalize masked logits into probabilities.
print("masked attention weights:\n", np.round(weights_a3, 3)) # Inspect that forbidden pairs get zero probability.
assert np.allclose(weights_a3[0, 2:], [0.0, 0.0], atol=1e-40) # Verify blocked wraparound weights vanish.

In [ ]:
plt.figure(figsize=(4, 3)) # Create a compact masked-attention heatmap.
plt.imshow(weights_a3, cmap="magma", vmin=0, vmax=1) # Visualize post-softmax probabilities.
plt.colorbar(label="attention weight") # Add probability scale.
plt.xticks(range(4), window_tokens_a3); plt.yticks(range(4), window_tokens_a3) # Label by token ids.
plt.title("Advanced 3: masked shifted attention") # Title the mask result.
plt.show() # Display the heatmap.

▶ What you'll see: attention probability is concentrated only within allowed blocks; forbidden wrap pairs are dark.

👀 Takeaway: masking before softmax is how shifted-window attention stays local in the real image, not cyclic.

### Advanced 4 — Build a tiny two-stage Swin pipeline

**Goal.** Combine window attention, shifted attention, and patch merging, because a Swin backbone is these operations repeated in stages. We build it in 5 steps.

In [ ]:
X_a4 = np.linspace(0, 1, 4 * 4 * 2).reshape(4, 4, 2) # Create a 4x4 feature map with two channels.
print("input shape:", X_a4.shape) # Inspect the first stage shape.

▶ What you'll see: a small feature map ready for local attention.

In [ ]:
def window_mean_a4(X_in_a4, win_in_a4): # Define a tiny stand-in for window attention: replace each token by its window mean.
    H_in_a4, W_in_a4, C_in_a4 = X_in_a4.shape # Read dimensions.
    windows_in_a4 = X_in_a4.reshape(H_in_a4 // win_in_a4, win_in_a4, W_in_a4 // win_in_a4, win_in_a4, C_in_a4).transpose(0, 2, 1, 3, 4) # Partition windows.
    means_in_a4 = windows_in_a4.mean(axis=(2, 3), keepdims=True) # Compute one local summary per window.
    mixed_in_a4 = np.broadcast_to(means_in_a4, windows_in_a4.shape) # Broadcast the summary back to all tokens in the window.
    return mixed_in_a4.transpose(0, 2, 1, 3, 4).reshape(H_in_a4, W_in_a4, C_in_a4) # Restore grid layout.
Y_fixed_a4 = window_mean_a4(X_a4, 2) # Apply fixed-window local mixing.
print("fixed mixed shape:", Y_fixed_a4.shape) # Inspect shape preservation.

In [ ]:
shifted_a4 = np.roll(np.roll(Y_fixed_a4, -1, axis=0), -1, axis=1) # Shift before the second local block.
Y_shifted_a4 = np.roll(np.roll(window_mean_a4(shifted_a4, 2), 1, axis=0), 1, axis=1) # Mix shifted windows and unshift back.
print("shifted mixed shape:", Y_shifted_a4.shape) # Inspect shape preservation after SW-MSA style mixing.
assert Y_shifted_a4.shape == X_a4.shape # Verify attention blocks keep resolution.

In [ ]:
concat_a4 = Y_shifted_a4.reshape(2, 2, 2, 2, 2).transpose(0, 2, 1, 3, 4).reshape(2, 2, 8) # Patch merge 2x2 neighborhoods.
proj_a4 = np.ones((8, 4)) / 8 # Simple projection from 4C to 2C.
stage2_a4 = concat_a4 @ proj_a4 # Produce the next stage.
print("stage 2 shape:", stage2_a4.shape) # Inspect the coarser, wider output.
assert stage2_a4.shape == (2, 2, 4) # Verify H/2, W/2, and 2C.

In [ ]:
plt.figure(figsize=(6, 2.8)) # Create a three-panel pipeline view.
plt.subplot(1, 3, 1); plt.imshow(X_a4[:, :, 0], cmap="viridis"); plt.title("input") # Show stage input.
plt.subplot(1, 3, 2); plt.imshow(Y_shifted_a4[:, :, 0], cmap="viridis"); plt.title("after blocks") # Show locally mixed output.
plt.subplot(1, 3, 3); plt.imshow(stage2_a4[:, :, 0], cmap="viridis"); plt.title("merged") # Show coarser next stage.
plt.suptitle("Advanced 4: tiny Swin-style stage") # Title the full pipeline.
plt.show() # Display the pipeline.

▶ What you'll see: attention blocks keep the 4×4 resolution, then patch merging creates a 2×2 next stage.

👀 Takeaway: Swin alternates local mixing at a fixed resolution with merging transitions between stages.

### Advanced 5 — Compare a flat ViT-style map with a Swin pyramid

**Goal.** Contrast fixed-resolution tokens with hierarchical stages, because Swin's backbone behavior is what makes it useful for dense tasks. We build it in 4 steps.

In [ ]:
start_side_a5 = 16 # Start with a 16x16 patch-token grid.
start_channels_a5 = 3 # Start with three feature channels.
vit_shapes_a5 = [(start_side_a5, start_side_a5, start_channels_a5) for _ in range(4)] # A flat ViT-style sequence keeps one grid shape.
print("flat shapes:", vit_shapes_a5) # Inspect the non-hierarchical baseline.

▶ What you'll see: the flat representation keeps the same token resolution at every stage.

In [ ]:
swin_shapes_a5 = [] # Store Swin-like pyramid shapes.
side_a5, ch_a5 = start_side_a5, start_channels_a5 # Initialize side length and channels.
for stage_a5 in range(4): # Build four stages.
    swin_shapes_a5.append((side_a5, side_a5, ch_a5)) # Store current stage.
    side_a5 = max(1, side_a5 // 2) # Patch merging halves spatial resolution.
    ch_a5 = ch_a5 * 2 # Patch merging widens channels.
print("Swin shapes:", swin_shapes_a5) # Inspect the hierarchy.
assert swin_shapes_a5[2] == (4, 4, 12) # Verify the third stage shape.

In [ ]:
vit_tokens_a5 = np.array([h_a5 * w_a5 for h_a5, w_a5, c_a5 in vit_shapes_a5]) # Count flat tokens.
swin_tokens_a5 = np.array([h_a5 * w_a5 for h_a5, w_a5, c_a5 in swin_shapes_a5]) # Count pyramid tokens.
swin_channels_a5 = np.array([c_a5 for h_a5, w_a5, c_a5 in swin_shapes_a5]) # Count pyramid channels.
print("ViT token counts:", vit_tokens_a5) # Inspect flat token counts.
print("Swin token counts:", swin_tokens_a5) # Inspect hierarchical token counts.

In [ ]:
plt.figure(figsize=(5, 3)) # Create a compact comparison plot.
plt.plot(vit_tokens_a5, marker="o", label="flat tokens") # Plot constant flat-token count.
plt.plot(swin_tokens_a5, marker="s", label="Swin tokens") # Plot shrinking Swin-token count.
plt.plot(swin_channels_a5, marker="^", label="Swin channels") # Plot widening channel count.
plt.title("Advanced 5: flat tokens vs Swin pyramid") # Title the backbone comparison.
plt.xlabel("stage") # Label stage axis.
plt.legend() # Show curve labels.
plt.show() # Display the plot.

▶ What you'll see: the flat model keeps token count fixed, while Swin reduces tokens and increases channel width.

👀 Takeaway: Swin is not just cheaper attention; it is a transformer shaped like a multi-scale vision backbone.

---

# Reference walkthrough — original compact notebook

The sections above build every idea from scratch with detailed steps and worked examples. Below is the original compact notebook for this lesson, kept as a concise reference and for its practice prompts.

Swin makes transformers behave like vision backbones by attending locally, shifting the windows, and building a hierarchy of larger visual tokens.

Swin starts from ViT patch tokens, then restores locality and pyramids for dense vision. Local windows reduce attention cost, shifted windows let neighbors communicate, and patch merging lowers resolution while increasing channels.

Save a copy to Drive to edit.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(7)

## The concept, built once: local windows and hierarchy

A $4\times4$ grid has $16\times16=256$ global attention pairs. Four $2\times2$ windows use

$$4\cdot4\cdot4=64$$

pairs, one quarter of the global cost, before shifted windows and patch merging pass information between regions.

In [ ]:
def softmax(x):
    x = np.asarray(x, dtype=float)
    shifted = x - x.max(axis=-1, keepdims=True)
    exp_x = np.exp(shifted)
    return exp_x / exp_x.sum(axis=-1, keepdims=True)

def partition_windows(grid, window):
    groups = []
    for r in range(0, grid.shape[0], window):
        for c in range(0, grid.shape[1], window):
            groups.append(grid[r:r + window, c:c + window].ravel())
    return groups

def patch_merge(tokens):
    merged = []
    for r in range(0, tokens.shape[0], 2):
        row = []
        for c in range(0, tokens.shape[1], 2):
            block = tokens[r:r + 2, c:c + 2, :].reshape(-1)
            row.append(block[:6])
        merged.append(row)
    return np.array(merged)

def swin_window_attention():
    global_pairs = 16 * 16
    window_pairs = 4 * 4 * 4
    values = np.array([[2.0, 0.0], [0.0, 3.0]])
    weights = np.array([[0.731, 0.269], [0.269, 0.731]])
    outputs = weights @ values
    grid = np.arange(16).reshape(4, 4)
    shifted = np.roll(grid, shift=-2, axis=1)
    shifted_group = partition_windows(shifted, 2)[0]
    tokens = np.arange(4 * 4 * 3).reshape(4, 4, 3)
    merged = patch_merge(tokens)
    return global_pairs, window_pairs, outputs, shifted_group, merged

global_pairs, window_pairs, outputs, shifted_group, merged = swin_window_attention()
assert global_pairs == 256
assert window_pairs == 64
assert np.allclose(np.round(outputs[0], 3), [1.462, 0.807])
assert np.allclose(np.round(outputs[1], 3), [0.538, 2.193])
assert 2 in shifted_group
assert 3 in shifted_group
assert merged.shape == (2, 2, 6)
print("pairs", global_pairs, window_pairs)
print("two-token outputs", np.round(outputs, 3))
print("shifted group", shifted_group.tolist())
print("merged shape", merged.shape)

## Reusable Swin-style features

The ladder featurizer averages local windows, then repeats after a one-pixel shift. It also adds coarse pooled cells to mimic the hierarchy used by dense backbones.

In [ ]:
def window_means(img, window, shift):
    shifted = np.roll(img, shift=shift, axis=(0, 1))
    feats = []
    for r in range(0, shifted.shape[0], window):
        for c in range(0, shifted.shape[1], window):
            block = shifted[r:r + window, c:c + window]
            feats.append(block.mean())
            feats.append(block.std())
    return np.array(feats)

def swin_featurize(img):
    local = window_means(img, 2, 0)
    shifted = window_means(img, 2, 1)
    pooled = window_means(img, 4, 0)
    global_stats = np.array([img.mean(), img.std()])
    return np.concatenate([local, shifted, pooled, global_stats])

## Dataset ladder: D1 to D5

The shared `cv_ladder.py` ladder gives one CPU-safe interface: every rung returns images `X` with shape `(n, 8, 8)` and labels `y`. The same featurizer is used on every rung, so the accuracy curve measures scaling rather than a changing task.

In [ ]:
"""
F6 (Vision) shared dataset ladder — D1..D5 of rising complexity, CPU-only and run-all-safe.

This is the canonical ladder inlined into the classification-style Part-7 notebooks. Every
rung returns (X, y) with X shape (n, 8, 8) float in [0, 1] and integer labels y, so one
featurizer + classifier can run unchanged across all five rungs (the "watch it scale" story).

D4/D5 load real MNIST / CIFAR-10 via torchvision when the download is available (as in Colab) —
offline they fall back to a harder synthetic set so run-all never fails. Code is written one
statement per line for readability.
"""

import numpy as np
from sklearn.datasets import load_digits
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


def _resize_to_8x8(img):
    """Nearest-neighbour resize of a 2-D array to 8x8 (no SciPy dependency)."""
    h, w = img.shape
    rows = (np.linspace(0, h - 1, 8)).round().astype(int)
    cols = (np.linspace(0, w - 1, 8)).round().astype(int)
    return img[np.ix_(rows, cols)]


def _normalize(x):
    """Scale an array into [0, 1] — a flat array becomes all zeros."""
    x = x.astype(float)
    lo = x.min()
    hi = x.max()
    if hi - lo < 1e-12:
        return np.zeros_like(x)
    return (x - lo) / (hi - lo)


def d1_hand_patches():
    """D1 — hand-built 4x4 patches, 2 classes: a vertical line (col 1) vs a horizontal line (row 1).

    Fixed positions with light jitter, so the two classes are cleanly separable and the
    mechanism is fully visible — the easy first rung.
    """
    rng = np.random.default_rng(0)
    images = []
    labels = []
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[:, 1] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(0)
    for _ in range(24):
        patch = rng.uniform(0.0, 0.15, size=(4, 4))
        patch[1, :] = rng.uniform(0.85, 1.0)
        images.append(_resize_to_8x8(patch))
        labels.append(1)
    return np.array(images), np.array(labels)


def d2_synthetic_shapes():
    """D2 — clean synthetic shapes on an 8x8 grid, 2 classes (square vs disc)."""
    rng = np.random.default_rng(1)
    yy, xx = np.mgrid[0:8, 0:8]
    images = []
    labels = []
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        img[2:6, 2:6] = 0.9
        images.append(img)
        labels.append(0)
    for _ in range(80):
        img = rng.uniform(0.0, 0.1, size=(8, 8))
        disc = (xx - 3.5) ** 2 + (yy - 3.5) ** 2 <= 4.0
        img[disc] = 0.9
        images.append(img)
        labels.append(1)
    return np.array(images), np.array(labels)


def d3_sklearn_digits():
    """D3 — real sklearn digits (native 8x8), 4 classes for a fast, honest multi-class rung."""
    digits = load_digits()
    keep = np.isin(digits.target, [0, 1, 2, 3])
    X = digits.images[keep]
    y = digits.target[keep]
    X = np.array([_normalize(img) for img in X])
    return X, y


def _synthetic_textured(n_per_class, n_classes, noise, seed):
    """A harder synthetic fallback: textured class prototypes at 8x8 with noise."""
    rng = np.random.default_rng(seed)
    protos = [rng.uniform(0.0, 1.0, size=(8, 8)) for _ in range(n_classes)]
    images = []
    labels = []
    for cls in range(n_classes):
        for _ in range(n_per_class):
            img = protos[cls] + rng.normal(0.0, noise, size=(8, 8))
            images.append(_normalize(img))
            labels.append(cls)
    return np.array(images), np.array(labels)


def _call_with_timeout(fn, seconds):
    """Run fn() but abort with TimeoutError after `seconds` (guards slow/hanging downloads)."""
    import signal

    def _raise(signum, frame):
        raise TimeoutError("download timed out")

    old = signal.signal(signal.SIGALRM, _raise)
    signal.alarm(seconds)
    try:
        return fn()
    finally:
        signal.alarm(0)
        signal.signal(signal.SIGALRM, old)


def _load_mnist_gray(classes, n_per_class, seed, shift=False, noise=0.0):
    """Load MNIST via torchvision, grayscale + resize to 8x8, subsample. Raises on failure.

    MNIST is a small (~11 MB) real dataset. CIFAR-10 is deliberately avoided (a 170 MB
    download breaks run-all-safety) — the harder D5 rung instead shifts and noises MNIST.
    """
    import torchvision

    ds = torchvision.datasets.MNIST(root="./data", train=True, download=True)
    rng = np.random.default_rng(seed)
    targets = np.array(ds.targets)
    images = []
    labels = []
    for cls in classes:
        idx = np.where(targets == cls)[0][:n_per_class]
        for i in idx:
            arr = np.asarray(ds[int(i)][0], dtype=float)
            small = _resize_to_8x8(arr)
            if shift:
                small = np.roll(small, rng.integers(-1, 2), axis=0)
                small = np.roll(small, rng.integers(-1, 2), axis=1)
            if noise:
                small = small + rng.normal(0.0, noise * 255.0, size=(8, 8))
            images.append(_normalize(small))
            labels.append(cls)
    return np.array(images), np.array(labels)


def d4_mnist_or_fallback():
    """D4 — real MNIST (4 clean classes) when downloadable — else a harder synthetic set."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3], 60, seed=4), 30)
        return (X, y), "MNIST (real)"
    except Exception:
        return _synthetic_textured(60, 4, noise=0.35, seed=4), "synthetic (offline fallback)"


def d5_mnist_hard_or_fallback():
    """D5 — real MNIST, more classes with shift + noise (distribution shift) — else hardest synthetic."""
    try:
        X, y = _call_with_timeout(lambda: _load_mnist_gray([0, 1, 2, 3, 4, 5], 60, seed=5, shift=True, noise=0.12), 30)
        return (X, y), "MNIST shifted+noisy (real, harder)"
    except Exception:
        return _synthetic_textured(60, 6, noise=0.6, seed=5), "synthetic (offline fallback)"


def load_ladder():
    """Return the five rungs as a list of (name, X, y). D4/D5 note whether real data loaded."""
    rungs = []
    rungs.append(("D1 hand patches", *d1_hand_patches()))
    rungs.append(("D2 synthetic shapes", *d2_synthetic_shapes()))
    rungs.append(("D3 sklearn digits", *d3_sklearn_digits()))
    (x4, y4), tag4 = d4_mnist_or_fallback()
    rungs.append((f"D4 {tag4}", x4, y4))
    (x5, y5), tag5 = d5_mnist_hard_or_fallback()
    rungs.append((f"D5 {tag5}", x5, y5))
    return rungs


def accuracy_with(featurize, X, y):
    """Map each image through featurize, train logistic regression, return held-out accuracy."""
    feats = np.array([featurize(img) for img in X])
    x_tr, x_te, y_tr, y_te = train_test_split(feats, y, test_size=0.4, random_state=0, stratify=y)
    clf = LogisticRegression(max_iter=2000)
    clf.fit(x_tr, y_tr)
    return clf.score(x_te, y_te)


if __name__ == "__main__":
    for name, X, y in load_ladder():
        print(f"{name:38s} X={X.shape} classes={sorted(set(y.tolist()))} acc(flatten)={accuracy_with(lambda im: im.ravel(), X, y):.3f}")


In [ ]:
rungs = load_ladder()

fig, axes = plt.subplots(1, 5, figsize=(11, 2.4))
for ax, (name, X, y) in zip(axes, rungs):
    ax.imshow(X[0], cmap="gray", vmin=0, vmax=1)
    ax.set_title(f"{name}\n{X.shape}, {len(set(y))} classes", fontsize=8)
    ax.axis("off")
plt.tight_layout()

for name, X, y in rungs:
    classes = sorted(set(y.tolist()))
    print(f"{name:34s} shape={X.shape} classes={classes}")

In [ ]:
swin_scores = []
for name, X, y in rungs:
    score = accuracy_with(swin_featurize, X, y)
    swin_scores.append(score)
    print(f"{name:34s} accuracy={score:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(11, 2.5))
for ax, (name, X, y) in zip(axes, rungs):
    features = window_means(X[0], 2, 1).reshape(4, 8)
    ax.imshow(features, aspect="auto", cmap="magma")
    ax.set_title(name.split()[0] + " shifted windows", fontsize=8)
    ax.axis("off")
plt.tight_layout()

plt.figure(figsize=(5, 3))
plt.plot(range(1, 6), swin_scores, marker="o")
plt.xticks(range(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(0, 1.05)
plt.ylabel("held-out accuracy")
plt.title("Swin-style local features across the ladder")
plt.grid(True, alpha=0.3)
plt.show()

## Pitfall on D5: shift without boundary handling

A naive roll wraps the right edge into the left edge. Real shifted-window attention masks those wrapped tokens, and patch merging keeps a feature pyramid for dense tasks.

In [ ]:
edge_image = np.zeros((8, 8))
edge_image[:, 0] = 1
wrapped = np.roll(edge_image, shift=2, axis=1)
mask = np.zeros_like(edge_image)
mask[:, 2:] = 1
masked = wrapped * mask
wrapped_leak = int(wrapped[:, 2].sum())
masked_leak = int(masked[:, 0].sum())
no_merge_score = accuracy_with(lambda img: np.array([img.mean(), img.std()]), rungs[-1][1], rungs[-1][2])
merge_score = accuracy_with(swin_featurize, rungs[-1][1], rungs[-1][2])
print("wrapped-edge activation", wrapped_leak)
print("masked left-edge activation", masked_leak)
print("without hierarchy accuracy", round(no_merge_score, 3))
print("with hierarchy accuracy", round(merge_score, 3))

## Evaluate it + Practice

- Metric: held-out accuracy; compare with the no-skill baseline, majority-class guessing.
- Cheap sanity check: D1 windows should separate line directions.
- Ablation: remove shifted windows or pooled hierarchy.
- Failure signals: edge wrapping leaks or accuracy falls on clutter.

Practice:
1. Change one D1 number and predict the metric before running it.

In [ ]:
# Your experiment here

2. Add one harder case to the ladder and rerun the curve.

In [ ]:
# Your harder case here

3. Turn off the key fix in the pitfall cell and explain the change.

In [ ]:
# Your ablation here